# LeetCode #158: Read N Characters Given Read4 II - Call multiple times

https://leetcode.com/problems/read-n-characters-given-read4-ii-call-multiple-times/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Buffered Read with Internal State** | $O(n)$ per call | $O(1)$ extra |

---

## Understanding the Methods

### Buffered Read with Internal State ★
Unlike problem #157, `read` can be called multiple times on the same file. The key challenge: a previous `read4` call may have fetched more characters than the previous `read` needed, so we must buffer leftover characters.

Maintain instance variables:
- `buf4[4]`: internal buffer holding the last `read4` result
- `buf4Count`: how many valid chars are in `buf4`
- `buf4Index`: current read position within `buf4`

For each `read(buf, n)` call:
1. First consume any leftover characters from `buf4`.
2. When `buf4` is exhausted, call `read4` again to refill.
3. Stop when `n` characters are copied or `read4` returns 0 (EOF).

**Key insight:** The internal buffer persists between calls, so leftover characters from the previous call are available for the next one.

**Constraints:**
* `read` may be called multiple times.

## Solutions

### C#

In [ ]:
public class Solution : Reader4 {
    private char[] buf4 = new char[4];
    private int buf4Count = 0;
    private int buf4Index = 0;
    
    public int Read(char[] buf, int n) {
        int total = 0;
        
        while (total < n) {
            // Refill internal buffer if exhausted
            if (buf4Index == buf4Count) {
                buf4Count = Read4(buf4);
                buf4Index = 0;
                if (buf4Count == 0) break; // EOF
            }
            // Copy from internal buffer
            while (total < n && buf4Index < buf4Count) {
                buf[total++] = buf4[buf4Index++];
            }
        }
        
        return total;
    }
}

### Python

In [ ]:
class Solution:
    def __init__(self):
        self.buf4 = [''] * 4
        self.buf4_count = 0
        self.buf4_index = 0
    
    def read(self, buf, n):
        total = 0
        
        while total < n:
            if self.buf4_index == self.buf4_count:
                self.buf4_count = read4(self.buf4)
                self.buf4_index = 0
                if self.buf4_count == 0:
                    break
            while total < n and self.buf4_index < self.buf4_count:
                buf[total] = self.buf4[self.buf4_index]
                total += 1
                self.buf4_index += 1
        
        return total

### Go

In [ ]:
var solution = func(read4 func([]byte) int) func([]byte, int) int {
    buf4 := make([]byte, 4)
    buf4Count := 0
    buf4Index := 0
    
    return func(buf []byte, n int) int {
        total := 0
        
        for total < n {
            if buf4Index == buf4Count {
                buf4Count = read4(buf4)
                buf4Index = 0
                if buf4Count == 0 {
                    break
                }
            }
            for total < n && buf4Index < buf4Count {
                buf[total] = buf4[buf4Index]
                total++
                buf4Index++
            }
        }
        
        return total
    }
}

### Rust

In [ ]:
struct Solution {
    buf4: [char; 4],
    buf4_count: usize,
    buf4_index: usize,
}

impl Solution {
    fn new() -> Self {
        Solution {
            buf4: [' '; 4],
            buf4_count: 0,
            buf4_index: 0,
        }
    }
    
    fn read(&mut self, buf: &mut [char], n: i32) -> i32 {
        let n = n as usize;
        let mut total = 0;
        
        while total < n {
            if self.buf4_index == self.buf4_count {
                self.buf4_count = read4(&mut self.buf4) as usize;
                self.buf4_index = 0;
                if self.buf4_count == 0 {
                    break;
                }
            }
            while total < n && self.buf4_index < self.buf4_count {
                buf[total] = self.buf4[self.buf4_index];
                total += 1;
                self.buf4_index += 1;
            }
        }
        
        total as i32
    }
}

## Example Scenarios

1. **Two calls, no leftover:** File = `"abcdefgh"`. `read(buf, 4)` returns 4 (`"abcd"`). `read(buf, 4)` returns 4 (`"efgh"`).

2. **Leftover from previous call:** File = `"abcde"`. `read(buf, 3)` returns 3 (`"abc"`), but `read4` fetched 4 chars — `'d'` is buffered. `read(buf, 3)` starts from buffered `'d'`, calls `read4` again for `'e'`, returns 2.

3. **Read more than available:** File = `"ab"`. `read(buf, 5)` returns 2. `read4` returns 2 (EOF), loop breaks.

4. **Single character reads:** File = `"abc"`. Three calls of `read(buf, 1)` return `'a'`, `'b'`, `'c'`. The first `read4` fetches all 3, subsequent calls consume from the buffer.

5. **Empty file:** File = `""`. `read(buf, 1)` returns 0 immediately since `read4` returns 0.

![image](attachment:image.png)